In [41]:
from ccdc import io
from ccdc.io import EntryReader

from collections import defaultdict
from itertools import islice

from tqdm import tqdm

import pandas as pd

import time

from concurrent.futures import ProcessPoolExecutor

In [42]:
csd = EntryReader('CSD')
csd_reader = io.EntryReader('CSD')

### 1. Extract all records to single CSV table

In [43]:
all_df = []

start = time.time()

total_entry = 500
for entry in tqdm(islice(csd, total_entry), total=total_entry, desc="Traitement"):

    # check organic
    is_organic = entry.is_organic

    # check num components
    num_component = len(entry.molecule.components)

    # check metal
    has_metal = any(atom.is_metal for atom in entry.molecule.atoms)

    # add result
    all_df.append({
        "ID":entry.identifier,
        "SMILES": entry.molecule.smiles,
        "IS_ORGANIC": is_organic,
        "HAS_METAL":has_metal,
        "NUM_COMPONENT": num_component,
    })

delta_time = round(time.time() - start, 1)

all_df = pd.DataFrame(all_df)
all_df.to_csv("csd_all.csv", index=False)

print(f"Sequential : Processed {len(all_df)//1000}k entries in {delta_time}s")

Traitement: 100%|████████████████████████████| 500/500 [00:02<00:00, 168.93it/s]

Sequential : Processed 0k entries in 3.0s


In [44]:
all_df

,ID,SMILES,IS_ORGANIC,HAS_METAL,NUM_COMPONENT
0,AABHTZ,CC(=O)NN1C=NN=C1N(N=Cc1c(Cl)cccc1Cl)C(C)=O,True,False,1
1,AACANI10,[OH2][Ni]123OC(=O)CN41CCCN2(CCC4)CC(=O)O3.O.O,False,True,3
2,AACANI11,[OH2][Ni]123OC(=O)CN41CCCN2(CCC4)CC(=O)O3.O.O,False,True,3
3,AACFAZ,COC1=C(C(OC1=O)c1ccccc1Cl)C(C)=NN=C(C)C1=C(OC)...,True,False,1
4,AACFAZ10,COC1=C(C(OC1=O)c1ccccc1Cl)C(C)=NN=C(C)C1=C(OC)...,True,False,1
...,...,...,...,...,...
495,ABAZPT,CC(=O)N1=NC(=[NH][Pt]21[NH]=C(N=N2C(C)=O)C(C)(...,False,True,1
496,ABAZUA,Cc1cc(C)c(c(C)c1)B1(c2ccccc2c2ccc(cn12)c1ccc(c...,True,False,1
497,ABAZUB,O.COc1cc(Nc2c(cnc3cc(OCCCN4CCN(C)CC4)c(OC)cc23...,True,False,2
498,ABAZUB01,COc1cc(Nc2c(cnc3cc(OCCCN4CCN(C)CC4)c(OC)cc23)C...,True,False,2


In [53]:
start = time.time()

def chunk_extraction(chunk_range):
    start, stop = chunk_range
    results = []
    with io.EntryReader('CSD') as reader:
        for i in range(start, stop):
            entry = reader[i]
            mol = entry.molecule
            results.append({
                "ID": entry.identifier,
                "SMILES": mol.smiles,
                "IS_ORGANIC": entry.is_organic,
                "HAS_METAL": any(atom.is_metal for atom in mol.atoms),
                "NUM_COMPONENT": len(mol.components),
            })
    return results

with io.EntryReader('CSD') as reader:
    NUMBER_ENTRIES = len(reader)

NUMBER_ENTRIES = 50000
NUM_WORKERS = 24
CHUNK_SIZE = 500
chunk_ranges = [(i, min(i + CHUNK_SIZE, NUMBER_ENTRIES)) for i in range(0, NUMBER_ENTRIES, CHUNK_SIZE)]
pbar = tqdm(total=NUMBER_ENTRIES, unit="entry", desc=f"Extraction ({NUM_WORKERS} workers)", smoothing=0.1)

with ProcessPoolExecutor(max_workers=NUM_WORKERS) as executor:
    all_results = []
    for chunk_results in executor.map(chunk_extraction, chunk_ranges):
        all_results.extend(chunk_results)
        pbar.update(len(chunk_results))
pbar.close()

delta_time = round(time.time() - start, 1)

all_df = pd.DataFrame(all_results)
all_df.to_csv("csd_all.csv", index=False)

print(f"Processed {len(all_df)//1000}k entries in {delta_time}s ({NUM_WORKERS} workers)")

Extraction (24 workers): 100%|███████| 50000/50000 [00:19<00:00, 2502.77entry/s]


Processed 50k entries in 20.0s (24 workers)


In [28]:
all_df

,ID,SMILES,IS_ORGANIC,HAS_METAL,NUM_COMPONENT
0,AABHTZ,CC(=O)NN1C=NN=C1N(N=Cc1c(Cl)cccc1Cl)C(C)=O,True,False,1
1,AACANI10,[OH2][Ni]123OC(=O)CN41CCCN2(CCC4)CC(=O)O3.O.O,False,True,3
2,AACANI11,[OH2][Ni]123OC(=O)CN41CCCN2(CCC4)CC(=O)O3.O.O,False,True,3
3,AACFAZ,COC1=C(C(OC1=O)c1ccccc1Cl)C(C)=NN=C(C)C1=C(OC)...,True,False,1
4,AACFAZ10,COC1=C(C(OC1=O)c1ccccc1Cl)C(C)=NN=C(C)C1=C(OC)...,True,False,1
...,...,...,...,...,...
499995,IYUYUY,CCCCCCC1c2cc3C(CCCCCC)c4cc5C(CCCCCC)c6cc7C(CCC...,True,False,6
499996,IYUYUZ,CC1=CNC(=O)NC1=O.O,True,False,2
499997,IYUYUZ01,CC1=CNC(=O)NC1=O.O,True,False,2
499998,IYUYUZ02,CC1=CNC(=O)NC1=O.O,True,False,2


### 2. Select only organic one component molecules

In [42]:
all_df = pd.read_csv("csd_all.csv")

df_filtered = all_df[
    (all_df["IS_ORGANIC"] == True) &
    (all_df["HAS_METAL"] == False) &
    (all_df["NUM_COMPONENT"] == 1)
]

print(f"Selected {len(df_filtered)} entries")

Selected 1423 entries


In [43]:
df_filtered

,ID,SMILES,IS_ORGANIC,HAS_METAL,NUM_COMPONENT
0,AABHTZ,CC(=O)NN1C=NN=C1N(N=Cc1c(Cl)cccc1Cl)C(C)=O,True,False,1
3,AACFAZ,COC1=C(C(OC1=O)c1ccccc1Cl)C(C)=NN=C(C)C1=C(OC)...,True,False,1
4,AACFAZ10,COC1=C(C(OC1=O)c1ccccc1Cl)C(C)=NN=C(C)C1=C(OC)...,True,False,1
6,AACMHX10,CC(=O)OC(=C1CCCCC1c1ccccc1)c1ccccc1,True,False,1
19,AAMAND,COC1CCC2(C)C(CCC3C2CCC2(C)C3CC2C(C)=O)C1,True,False,1
...,...,...,...,...,...
4978,ACIZES,CC(C)c1cc(C(C)C)c(c(c1)C(C)C)[Si](=P)[Si](C(C)...,True,False,1
4979,ACIZET,FC(F)(F)c1cccc(NC(=S)NC2CCCCC2)c1,True,False,1
4989,ACIZOC,CC(C)[Si]1(O[Si](P[Si](O[Si](P1)(C(C)C)C(C)C)(...,True,False,1
4990,ACIZOD,CC1=NN(c2ccccc2)C2=C1C1(C)C(COc3ccccc13)CO2,True,False,1


### 3. Count the number of crystal forms

In [44]:
groups = defaultdict(list)
for entry_id in tqdm(df_filtered["ID"]):

    mol = csd_reader.molecule(entry_id)
    key = mol.generate_inchi().inchi

    if key:  # skip if missing
        groups[key].append(mol.identifier)

print(f"Selected {len(groups)} molecules with InChIKey")

100%|██████████████████████████████████████████████████████████████████████████████| 1423/1423 [00:03<00:00, 456.10it/s]

Selected 1192 molecules with InChIKey


In [45]:
df_filtered = df_filtered.set_index("ID")

df_counted = []

for key, entries in groups.items():
    num_forms = len(entries)

    df_counted.append({
        "SMILES": df_filtered.loc[entries[0]]["SMILES"],
        "NUM_FORMS": len(entries)
    })

df_counted = pd.DataFrame(df_counted)
df_counted = df_counted.drop_duplicates("SMILES")

print(f"Selected {len(df_counted)} molecules with counted forms")

Selected 1175 molecules with counted forms


In [46]:
n_total = len(df_counted)
n_mono = sum(df_counted["NUM_FORMS"] == 1)
n_poly = sum(df_counted["NUM_FORMS"] > 1)

print(f"Total molecules: {n_total}")
print(f"Monomorphs: {n_mono} ({round(100 * n_mono / n_total, 1)} %)")
print(f"Polymorphs: {n_poly} ({round(100 * n_poly / n_total, 1)} %)")

Total molecules: 1175
Monomorphs: 1104 (94.0 %)
Polymorphs: 71 (6.0 %)


In [48]:
df_counted.to_csv("csd_counted.csv", index=False)